In [ ]:
!pip install -q kaggle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

In [ ]:
# OPTION A: Download directly via Kaggle API
# 1. Go to kaggle.com -> Account -> Create New API Token -> downloads kaggle.json
# 2. Upload that file when prompted below

# from google.colab import files
# print("Upload your kaggle.json file (from Kaggle Account settings)")
# uploaded = files.upload()

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sudalairajkumar/indian-startup-funding -p /content --unzip

In [ ]:
# Adjust filename if different after extraction
df = pd.read_csv("/content/startup_funding.csv")

print("Shape:", df.shape)
df.head()

Initial Inspection


In [ ]:
print(df.info())
print("\nColumn Names:\n", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

Data Cleaning & Preprocessing

In [ ]:
# Standardize column names
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
print(df.columns.tolist())

# Common columns in this dataset:
# Sr_No, Date_dd/mm/yyyy, Startup_Name, Industry_Vertical, SubVertical,
# City__Location, Investors_Name, InvestmentnType, Amount_in_USD, Remarks

# --- 1. Fix Date column ---
date_col = [c for c in df.columns if 'Date' in c][0]

# Fix known malformed dates (common issues in this dataset)
df[date_col] = df[date_col].astype(str).str.strip()
df[date_col] = df[date_col].str.replace('.', '/', regex=False)
df[date_col] = df[date_col].str.replace('//', '/', regex=False)

# Fix specific known typos in this dataset
df[date_col] = df[date_col].replace({
    '12/05.2015': '12/05/2015',
    '13/04.2015': '13/04/2015',
    '15/01.2015': '15/01/2015',
    '22/01//2015': '22/01/2015',
    '05/072018': '05/07/2018',
    '01/07/2015': '01/07/2015',
    '\\\\xc2\\\\xa0': ''
})

df['Date_clean'] = pd.to_datetime(df[date_col], format='%d/%m/%Y', errors='coerce')
print("Unparsed dates:", df['Date_clean'].isnull().sum())

df['Year'] = df['Date_clean'].dt.year
df['Month'] = df['Date_clean'].dt.month
df['YearMonth'] = df['Date_clean'].dt.to_period('M')

# --- 2. Clean Amount column ---
amt_col = [c for c in df.columns if 'Amount' in c][0]

df[amt_col] = df[amt_col].astype(str).str.replace(',', '', regex=False)
df[amt_col] = df[amt_col].str.replace('+', '', regex=False)
df[amt_col] = df[amt_col].str.replace('undisclosed', 'NaN', case=False, regex=False)
df[amt_col] = df[amt_col].str.replace('unknown', 'NaN', case=False, regex=False)
df[amt_col] = df[amt_col].str.replace('Undisclosed', 'NaN', regex=False)
df[amt_col] = df[amt_col].str.extract(r'(\d+\.?\d*)')[0]
df['Amount_USD'] = pd.to_numeric(df[amt_col], errors='coerce')

# Drop extreme outliers/typos (e.g., amount entered with extra zeros)
upper_limit = df['Amount_USD'].quantile(0.99)
print("99th percentile funding amount:", upper_limit)

# --- 3. Clean text/categorical columns ---
text_cols = ['Startup_Name', 'Industry_Vertical', 'SubVertical',
             'City__Location', 'Investors_Name', 'InvestmentnType']
text_cols = [c for c in text_cols if c in df.columns]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})
    df[col] = df[col].str.title()

# Standardize city names (common variants)
if 'City__Location' in df.columns:
    df['City__Location'] = df['City__Location'].replace({
        'Bangalore': 'Bengaluru',
        'Delhi': 'New Delhi',
        'Gurgaon': 'Gurugram'
    })

# Standardize investment types
if 'InvestmentnType' in df.columns:
    df['InvestmentnType'] = df['InvestmentnType'].replace({
        'Seed/ Angel Funding': 'Seed/Angel Funding',
        'Seed / Angel Funding': 'Seed/Angel Funding',
        'Seed\\Angel Funding': 'Seed/Angel Funding',
        'Privateequity': 'Private Equity',
        'Private  Equity': 'Private Equity'
    })

# Fill critical nulls
df['Industry_Vertical'] = df['Industry_Vertical'].fillna('Unknown')
df['City__Location'] = df['City__Location'].fillna('Unknown')
df['InvestmentnType'] = df['InvestmentnType'].fillna('Unknown')

print("\nCleaned dataset shape:", df.shape)
df.head()

Funding Trends Over Time

In [ ]:
yearly_funding = df.groupby('Year')['Amount_USD'].agg(['sum', 'count']).reset_index()
yearly_funding.columns = ['Year', 'Total_Funding_USD', 'Deal_Count']
yearly_funding = yearly_funding.dropna(subset=['Year'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=yearly_funding, x='Year', y='Total_Funding_USD', hue='Year', palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Total Funding Amount by Year')
axes[0].set_ylabel('Total Funding (USD)')

sns.lineplot(data=yearly_funding, x='Year', y='Deal_Count', marker='o', ax=axes[1], color='darkorange')
axes[1].set_title('Number of Funding Deals by Year')
axes[1].set_ylabel('Deal Count')

plt.tight_layout()
plt.show()

# Monthly trend
monthly = df.groupby('YearMonth')['Amount_USD'].sum().reset_index()
monthly['YearMonth'] = monthly['YearMonth'].astype(str)

plt.figure(figsize=(16, 5))
plt.plot(monthly['YearMonth'], monthly['Amount_USD'], color='teal')
plt.xticks(rotation=90, fontsize=7)
plt.title('Monthly Funding Trend')
plt.ylabel('Funding Amount (USD)')
plt.tight_layout()
plt.show()

Top Sectors (Industry Verticals)

In [ ]:
top_sectors_amt = df.groupby('Industry_Vertical')['Amount_USD'].sum().sort_values(ascending=False).head(15)
top_sectors_count = df['Industry_Vertical'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.barplot(x=top_sectors_amt.values, y=top_sectors_amt.index, hue=top_sectors_amt.index, palette='mako', legend=False, ax=axes[0])
axes[0].set_title('Top 15 Sectors by Total Funding Amount')
axes[0].set_xlabel('Total Funding (USD)')

sns.barplot(x=top_sectors_count.values, y=top_sectors_count.index, hue=top_sectors_count.index, palette='rocket', legend=False, ax=axes[1])
axes[1].set_title('Top 15 Sectors by Number of Deals')
axes[1].set_xlabel('Number of Deals')

plt.tight_layout()
plt.show()

Top Cities

In [ ]:
top_cities = df['City__Location'].value_counts().head(15)
top_cities_amt = df.groupby('City__Location')['Amount_USD'].sum().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.barplot(x=top_cities.values, y=top_cities.index, hue=top_cities.index, palette='crest', legend=False, ax=axes[0])
axes[0].set_title('Top 15 Cities by Number of Startups Funded')

sns.barplot(x=top_cities_amt.values, y=top_cities_amt.index, hue=top_cities_amt.index, palette='flare', legend=False, ax=axes[1])
axes[1].set_title('Top 15 Cities by Total Funding Amount')

plt.tight_layout()
plt.show()

Top Startups by Total Funding

In [ ]:
top_startups = df.groupby('Startup_Name')['Amount_USD'].sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=top_startups.values, y=top_startups.index, hue=top_startups.index, palette='magma', legend=False)
plt.title('Top 15 Startups by Total Funding Raised')
plt.xlabel('Total Funding (USD)')
plt.tight_layout()
plt.show()

Active Investors

In [ ]:
# Investors_Name often contains multiple comma-separated investors -> explode
investors_series = df['Investors_Name'].dropna().str.split(',')
all_investors = investors_series.explode().str.strip()
all_investors = all_investors[all_investors.str.lower() != 'undisclosed investors']
all_investors = all_investors[all_investors != '']

top_investors = all_investors.value_counts().head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=top_investors.values, y=top_investors.index, hue=top_investors.index, palette='cubehelix', legend=False)
plt.title('Top 15 Most Active Investors (by Number of Deals)')
plt.xlabel('Number of Investments')
plt.tight_layout()
plt.show()

 Investment Type Distribution

In [ ]:
inv_type_counts = df['InvestmentnType'].value_counts().head(10)

plt.figure(figsize=(10, 8))
plt.pie(inv_type_counts.values, labels=inv_type_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('Set2'), startangle=90)
plt.title('Distribution of Investment Types')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x=inv_type_counts.values, y=inv_type_counts.index, hue=inv_type_counts.index, palette='Set2', legend=False)
plt.title('Investment Type Frequency')
plt.xlabel('Number of Deals')
plt.tight_layout()
plt.show()

Funding Amount Distribution (Boxplot/Outliers)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x=df['Amount_USD'].dropna(), color='skyblue')
plt.title('Distribution of Funding Amounts (USD) — Outlier Check')
plt.xlabel('Amount (USD)')
plt.show()

# Log-scale histogram for skewed funding amounts
plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(df['Amount_USD'].dropna()), bins=40, kde=True, color='purple')
plt.title('Log-Transformed Distribution of Funding Amounts')
plt.xlabel('log(1 + Amount USD)')
plt.show()

 Sector vs City Heatmap

In [ ]:
top_sector_list = df['Industry_Vertical'].value_counts().head(10).index
top_city_list = df['City__Location'].value_counts().head(10).index

pivot = df[df['Industry_Vertical'].isin(top_sector_list) & df['City__Location'].isin(top_city_list)]
pivot_table = pivot.pivot_table(index='Industry_Vertical', columns='City__Location',
                                 values='Amount_USD', aggfunc='count', fill_value=0)

plt.figure(figsize=(14, 8))
sns.heatmap(pivot_table, annot=True, fmt='g', cmap='YlGnBu')
plt.title('Number of Deals: Top Sectors vs Top Cities')
plt.tight_layout()
plt.show()

Summary Statistics Table

In [ ]:
summary = {
    "Total Funding Deals": len(df),
    "Total Funding Amount (USD)": df['Amount_USD'].sum(),
    "Average Deal Size (USD)": df['Amount_USD'].mean(),
    "Median Deal Size (USD)": df['Amount_USD'].median(),
    "Unique Startups": df['Startup_Name'].nunique(),
    "Unique Investors": all_investors.nunique(),
    "Date Range": f"{df['Date_clean'].min().date()} to {df['Date_clean'].max().date()}"
}

summary_df = pd.DataFrame(summary.items(), columns=['Metric', 'Value'])
summary_df

## Predictive Model: Estimating Funding Amount

We train a regression model to predict `Amount_USD` (log-transformed) from `Industry_Vertical`, `City__Location`, `InvestmentnType`, and `Year`. The trained model + preprocessing pipeline is then saved to disk with `joblib` so it can be reloaded later without retraining.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# --- Prepare modeling dataframe ---
model_df = df.dropna(subset=['Amount_USD', 'Year']).copy()

feature_cols = ['Industry_Vertical', 'City__Location', 'InvestmentnType', 'Year']
target_col = 'Amount_USD'

model_df = model_df.dropna(subset=feature_cols)

X = model_df[feature_cols].copy()
X['Year'] = X['Year'].astype(int)
y = np.log1p(model_df[target_col])  # log-transform target (right-skewed amounts)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

# --- Preprocessing + model pipeline ---
categorical_features = ['Industry_Vertical', 'City__Location', 'InvestmentnType']
numeric_features = ['Year']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ],
    remainder='passthrough'  # keeps 'Year' as-is
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)

# --- Evaluate ---
y_pred_log = model_pipeline.predict(X_test)
mae_log = mean_absolute_error(y_test, y_pred_log)
r2 = r2_score(y_test, y_pred_log)

# Also report MAE back in original USD scale for interpretability
y_test_usd = np.expm1(y_test)
y_pred_usd = np.expm1(y_pred_log)
mae_usd = mean_absolute_error(y_test_usd, y_pred_usd)

print(f"R^2 score (log scale): {r2:.4f}")
print(f"MAE (log scale): {mae_log:.4f}")
print(f"MAE (original USD scale): ${mae_usd:,.0f}")

### Feature Importance

In [ ]:
ohe = model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
ohe_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = list(ohe_feature_names) + numeric_features

importances = model_pipeline.named_steps['regressor'].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x=feat_imp.values, y=feat_imp.index, hue=feat_imp.index, palette='viridis', legend=False)
plt.title('Top 20 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Saving the Trained Model

We persist the full pipeline (preprocessing + regressor) to a single `.joblib` file, along with the feature column list and target transform info, so it can be loaded and used for inference later without re-running the training code.

In [ ]:
import joblib
import os

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "startup_funding_model.joblib")

model_artifact = {
    "pipeline": model_pipeline,
    "feature_cols": feature_cols,
    "target_col": target_col,
    "target_transform": "log1p",  # remember to apply np.expm1() to predictions
    "metrics": {
        "r2_log_scale": r2,
        "mae_log_scale": mae_log,
        "mae_usd_scale": mae_usd
    }
}

joblib.dump(model_artifact, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")
print(f"File size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

### Reloading & Using the Saved Model (Inference Example)

In [ ]:
# Demonstrates loading the saved model and using it for a new prediction
loaded_artifact = joblib.load(MODEL_PATH)
loaded_pipeline = loaded_artifact["pipeline"]

sample = pd.DataFrame([{
    "Industry_Vertical": "Technology",
    "City__Location": "Bengaluru",
    "InvestmentnType": "Seed/Angel Funding",
    "Year": 2018
}])

pred_log = loaded_pipeline.predict(sample)
pred_usd = np.expm1(pred_log)[0]

print(f"Predicted funding amount: ${pred_usd:,.0f}")

Insights & Recommendations (Markdown Cell)

## 📊 Key Insights

1. **Funding Growth Pattern**: Total deal volume and capital deployed show distinct yearly trends —
   identify peak years to understand macro investment cycles (e.g., funding winters vs booms).
2. **Sector Concentration**: A small number of sectors (e.g., Consumer Internet, Technology, E-commerce,
   Healthcare) attract a disproportionate share of capital — indicating where investor confidence concentrates.
3. **City Concentration**: Funding is heavily concentrated in a few metro hubs (Bengaluru, Mumbai,
   New Delhi/Gurugram) — tier-2/3 cities remain underfunded relative to their startup activity.
4. **Investor Activity**: A core group of repeat investors drives a large share of deal volume,
   suggesting strong "anchor investor" patterns and possible co-investment networks.
5. **Investment Type Skew**: Seed/Angel funding dominates deal count, while later-stage rounds
   (Private Equity, Series rounds) dominate total capital — typical of an early-stage-heavy ecosystem.
6. **High Variance in Deal Size**: Funding amounts are highly right-skewed; a few mega-deals
   significantly inflate total funding statistics, so median figures are more representative of "typical" deals.

## 💡 Recommendations

### For Investors
- **Diversify beyond metro hubs**: Emerging tier-2 cities (Pune, Jaipur, Chandigarh) show rising deal
  activity at lower valuations — early entry could yield better returns.
- **Track sector cycles**: Allocate capital toward sectors showing renewed momentum in recent years
  rather than historically dominant ones that may be saturated.
- **Co-invest with active anchor investors**: Following top active investors' portfolios can reduce
  due-diligence risk and improve deal-flow quality.
- **Watch investment-type mix**: A shift toward growth-stage rounds in a sector can be an early signal
  of ecosystem maturity in that space.

### For Startup Founders
- **Sector positioning matters**: Founders in saturated sectors should differentiate clearly on
  business model or target city to stand out for funding.
- **Consider non-metro bases**: Lower competition for capital exists outside top funded cities,
  along with lower operating costs — useful for runway extension.
- **Target the right investment type at the right stage**: Seed-stage founders should focus pitches
  toward angel/seed-focused investors; later-stage founders should research investors with a track
  record in PE/Series rounds.
- **Benchmark realistic ask sizes**: Use median deal size (not mean, which is skewed by mega-deals)
  for your sector/city combination when setting fundraising targets.

## Interactive Gradio UI for Funding Prediction

In [ ]:
# Install Gradio (quiet mode). Skip this cell if already installed.
!pip install -q gradio


In [ ]:
import gradio as gr
import joblib
import numpy as np

# Reuses the model saved earlier in the notebook (MODEL_PATH / model_pipeline).
# If you're running this section on its own (fresh runtime), load it directly:
#   MODEL_PATH = "/content/models/startup_funding_model.joblib"
loaded_artifact = joblib.load(MODEL_PATH)
ui_pipeline = loaded_artifact["pipeline"]

# Dropdown choices pulled straight from the cleaned dataframe so the UI
# only ever offers categories the model actually saw during training.
industry_choices = sorted(df['Industry_Vertical'].dropna().unique().tolist())
city_choices = sorted(df['City__Location'].dropna().unique().tolist())
invtype_choices = sorted(df['InvestmentnType'].dropna().unique().tolist())
year_min, year_max = int(df['Year'].min()), int(df['Year'].max())


def predict_funding(industry, city, investment_type, year):
    """
    Takes the four model features from the Gradio inputs, runs them through
    the saved RandomForest pipeline, and returns the predicted funding amount
    in USD (undoing the log1p transform the model was trained on).
    """
    sample = pd.DataFrame([{
        "Industry_Vertical": industry,
        "City__Location": city,
        "InvestmentnType": investment_type,
        "Year": int(year),
    }])

    pred_log = ui_pipeline.predict(sample)
    pred_usd = float(np.expm1(pred_log)[0])

    return f"${pred_usd:,.0f}"


demo = gr.Interface(
    fn=predict_funding,
    inputs=[
        gr.Dropdown(choices=industry_choices, label="Industry Vertical", value=industry_choices[0]),
        gr.Dropdown(choices=city_choices, label="City / Location", value=city_choices[0]),
        gr.Dropdown(choices=invtype_choices, label="Investment Type", value=invtype_choices[0]),
        gr.Slider(minimum=year_min, maximum=year_max, step=1, value=year_max, label="Year"),
    ],
    outputs=gr.Textbox(label="Predicted Funding Amount"),
    title="Indian Startup Funding Predictor",
    description=(
        "Pick an industry, city, investment type, and year to get the RandomForest "
        "model's estimated funding amount (USD), trained on the Indian Startup "
        "Funding dataset."
    ),
    flagging_mode="never",
)

# share=True gives you a public URL (useful in Colab); set to False for local-only.
demo.launch(share=True, debug=False)
